# 04 CNN Baseline Training

## Objective
Train a custom CNN baseline model:
- ✓ Build custom CNN architecture
- ✓ Load preprocessed data
- ✓ Train model with validation
- ✓ Monitor training with callbacks
- ✓ Save training history
- ✓ Evaluate on test set
- ✓ Plot training curves

**Prerequisites:** `01_data_check.ipynb`, `02_dataset_visualization.ipynb`, `03_data_preprocessing.ipynb`

### Import Libraries

In [ ]:
import sys
import os

sys.path.insert(0, '../utils')

from model_utils import (
    build_cnn_model,
    train_model,
    evaluate_model,
    save_model,
    save_metrics,
    get_model_summary,
    compute_class_weights_from_labels
)
from visualization_utils import (
    plot_training_history,
    plot_confusion_matrix,
    plot_metrics_comparison
)

import numpy as np
import tensorflow as tf
from tensorflow import keras
import json
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

# Set random seeds
np.random.seed(42)
tf.random.set_seed(42)

print("✓ Libraries imported successfully!")
print(f"TensorFlow version: {tf.__version__}")


✓ Libraries imported successfully!
TensorFlow version: 2.15.0


### Load Preprocessed Data

In [2]:
print("Loading preprocessed data...\n")

# Load data
X_train = np.load('../results/preprocessed_data/X_train.npy')
y_train = np.load('../results/preprocessed_data/y_train.npy')
X_val = np.load('../results/preprocessed_data/X_val.npy')
y_val = np.load('../results/preprocessed_data/y_val.npy')
X_test = np.load('../results/preprocessed_data/X_test.npy')
y_test = np.load('../results/preprocessed_data/y_test.npy')

# Load label encoding
with open('../results/preprocessed_data/label_encoding.json', 'r') as f:
    label_encoding = json.load(f)

num_classes = len(label_encoding)

print(f"✓ Data loaded successfully")
print(f"\nData shapes:")
print(f"  X_train: {X_train.shape}")
print(f"  y_train: {y_train.shape}")
print(f"  X_val: {X_val.shape}")
print(f"  y_val: {y_val.shape}")
print(f"  X_test: {X_test.shape}")
print(f"  y_test: {y_test.shape}")
print(f"\nNumber of classes: {num_classes}")

Loading preprocessed data...

✓ Data loaded successfully

Data shapes:
  X_train: (532, 224, 224, 3)
  y_train: (532, 40)
  X_val: (114, 224, 224, 3)
  y_val: (114, 40)
  X_test: (115, 224, 224, 3)
  y_test: (115, 40)

Number of classes: 40


In [ ]:
class_names = [label for label, _ in sorted(label_encoding.items(), key=lambda item: item[1])]
class_weight = compute_class_weights_from_labels(y_train, class_names=class_names, verbose=True)


### Configuration

In [3]:
# Training configuration
EPOCHS = 80
BATCH_SIZE = 16
LEARNING_RATE = 0.0005
WEIGHT_DECAY = 1e-4
MODEL_NAME = 'cnn_baseline'

print("Training Configuration:")
print(f"  Model: Custom CNN")
print(f"  Epochs: {EPOCHS}")
print(f"  Batch Size: {BATCH_SIZE}")
print(f"  Learning Rate: {LEARNING_RATE}")
print(f"  Weight Decay: {WEIGHT_DECAY}")

Training Configuration:
  Model: Custom CNN
  Epochs: 50
  Batch Size: 32
  Learning Rate: 0.001


### 1. Build CNN Model

In [ ]:
print("Building CNN model...\n")

input_shape = (224, 224, 3)
model = build_cnn_model(
    num_classes=num_classes,
    input_shape=input_shape,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

print("✓ Model built successfully!\n")
print(get_model_summary(model))

### 2. Setup Callbacks

In [4]:
# Create callbacks
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=8,
        restore_best_weights=True,
        verbose=1,
        mode='min'
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-6,
        verbose=1,
        mode='min'
    ),
    keras.callbacks.ModelCheckpoint(
        '../models/cnn/best_model.h5',
        monitor='val_loss',
        save_best_only=True,
        verbose=0,
        mode='min'
    )
]

os.makedirs('../models/cnn', exist_ok=True)

print("✓ Callbacks configured")
print("  - Early Stopping (patience=8)")
print("  - Learning Rate Reduction (factor=0.5)")
print("  - Model Checkpoint (save best val_loss)")

✓ Callbacks configured
  - Early Stopping (patience=10)
  - Learning Rate Reduction (factor=0.5)
  - Model Checkpoint (save best)


### 3. Train Model

In [ ]:
print("\nTraining model...\n")
print("="*70)

history = train_model(
    model=model,
    train_data=(X_train, y_train),
    val_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    class_weight=class_weight
)

print("\n" + "="*70)
print("✓ Training completed!")


Training model...



NameError: name 'model' is not defined

### 4. Plot Training History

In [ ]:
print("Plotting training history...\n")

plot_training_history(history, figsize=(14, 5))
plt.savefig('../results/plots/cnn_training_history.png', dpi=300, bbox_inches='tight')
print("✓ Saved: cnn_training_history.png")

### 5. Evaluate on Test Set

In [ ]:
print("\nEvaluating model on test set...\n")

# Reverse label encoding for class names
reverse_encoding = {v: k for k, v in label_encoding.items()}
class_names = [reverse_encoding[i] for i in range(num_classes)]

# Evaluate
metrics = evaluate_model(model, X_test, y_test, class_names)

print("\n" + "="*70)
print("MODEL EVALUATION RESULTS")
print("="*70)
print(f"\nAccuracy:  {metrics['accuracy']:.4f}")
print(f"Precision: {metrics['precision']:.4f}")
print(f"Recall:    {metrics['recall']:.4f}")
print(f"F1-Score:  {metrics['f1_score']:.4f}")

### 6. Plot Confusion Matrix

In [ ]:
print("\nPlotting confusion matrix...\n")

cm = np.array(metrics['confusion_matrix'])
plot_confusion_matrix(cm, class_names, figsize=(14, 12), normalize=False)
plt.savefig('../results/plots/cnn_confusion_matrix.png', dpi=300, bbox_inches='tight')
print("✓ Saved: cnn_confusion_matrix.png")

# Normalized
plot_confusion_matrix(cm, class_names, figsize=(14, 12), normalize=True)
plt.savefig('../results/plots/cnn_confusion_matrix_normalized.png', dpi=300, bbox_inches='tight')
print("✓ Saved: cnn_confusion_matrix_normalized.png")

### 7. Plot Metrics Comparison

In [ ]:
metrics_dict = {
    'Accuracy': metrics['accuracy'],
    'Precision': metrics['precision'],
    'Recall': metrics['recall'],
    'F1-Score': metrics['f1_score']
}

plot_metrics_comparison(metrics_dict, figsize=(10, 6))
plt.savefig('../results/plots/cnn_metrics.png', dpi=300, bbox_inches='tight')
print("✓ Saved: cnn_metrics.png")

### 8. Save Model

In [ ]:
print("\nSaving model...\n")

model_path = save_model(model, MODEL_NAME, output_dir='../models/cnn')

print(f"\n✓ Model saved to: {model_path}")

### 9. Save Metrics and History

In [ ]:
print("Saving metrics and history...\n")

# Save metrics
metrics_path = save_metrics(metrics, 'cnn_baseline_metrics.json', output_dir='../results/reports')

# Save history
with open('../results/reports/cnn_baseline_history.json', 'w') as f:
    # Convert numpy values to float for JSON serialization
    history_serializable = {k: [float(v) for v in vals] for k, vals in history.items()}
    json.dump(history_serializable, f, indent=4)

print("✓ Metrics saved")
print("✓ History saved")

### 10. Summary

In [ ]:
print("\n" + "="*70)
print("✓ CNN BASELINE TRAINING COMPLETE")
print("="*70)

print(f"\n📊 Results Summary:")
print(f"  Model: Custom CNN")
print(f"  Test Accuracy: {metrics['accuracy']:.4f}")
print(f"  Test Precision: {metrics['precision']:.4f}")
print(f"  Test Recall: {metrics['recall']:.4f}")
print(f"  Test F1-Score: {metrics['f1_score']:.4f}")

print(f"\n💾 Files Saved:")
print(f"  - Model: ../models/cnn/cnn_baseline.h5")
print(f"  - Best Model: ../models/cnn/best_model.h5")
print(f"  - Metrics: ../results/reports/cnn_baseline_metrics.json")
print(f"  - History: ../results/reports/cnn_baseline_history.json")
print(f"  - Plots: ../results/plots/")

print(f"\n→ Next Step: 05_mobilenet_training.ipynb")